# 结构化输出（Structured Outputs）简介

Structured Outputs 是 Chat Completions API 和 Assistants API 中的一项新功能，它保证模型始终生成符合你所提供的 JSON Schema 的响应。在本 cookbook 中，我们将通过几个示例来演示这一功能。

在 API 调用中，无论是使用定义好的响应格式还是函数定义，都可以通过设置参数 `strict: true` 来启用 Structured Outputs。

## 响应格式用法

以前，`response_format` 参数只能用于指定模型应返回有效的 JSON。

除此之外，我们现在引入了一种新方式，用于指定要遵循的 JSON schema。


## 函数调用用法

函数调用的方式基本不变，但借助新参数 `strict: true`，你现在可以确保严格遵循为函数提供的 schema。


## 示例 

Structured Outputs 在很多方面都很有用，因为你可以依赖输出遵循受约束的 schema。

与 JSON 模式不同，Structured Outputs 约束的是 JSON 的结构，而不仅仅是其有效性。

无论你是依赖函数调用，还是期望输出遵循预定义的结构，这都能让生产级应用中的流程更加健壮。

示例用例包括：

- 获取结构化答案，以便在 UI 中以特定方式展示（本 cookbook 中的示例 1）
- 用从文档中抽取的内容填充数据库（本 cookbook 中的示例 2）
- 从用户输入中抽取实体，以使用定义好的参数调用工具（本 cookbook 中的示例 3）

更广泛地说，任何需要获取数据、执行操作或基于复杂工作流构建的场景，都可能从使用 Structured Outputs 中受益。

### 准备工作

In [ ]:
%pip install openai -U

In [1]:
import json
from textwrap import dedent
from openai import OpenAI
client = OpenAI()

In [2]:
MODEL = "gpt-4o-2024-08-06"

## 示例 1：数学辅导

在这个示例中，我们希望构建一个数学辅导工具，以结构化对象数组的形式输出解决数学问题的步骤。

这在需要分别展示每个步骤的应用中很有用，用户可以按照自己的节奏逐步查看解答。

In [3]:
math_tutor_prompt = '''
    You are a helpful math tutor. You will be provided with a math problem,
    and your goal will be to output a step by step solution, along with a final answer.
    For each step, just provide the output as an equation use the explanation field to detail the reasoning.
'''

def get_math_solution(question):
    response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system", 
            "content": dedent(math_tutor_prompt)
        },
        {
            "role": "user", 
            "content": question
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "math_reasoning",
            "schema": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "explanation": {"type": "string"},
                                "output": {"type": "string"}
                            },
                            "required": ["explanation", "output"],
                            "additionalProperties": False
                        }
                    },
                    "final_answer": {"type": "string"}
                },
                "required": ["steps", "final_answer"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
    )

    return response.choices[0].message

In [4]:
# Testing with an example question
question = "how can I solve 8x + 7 = -23"

result = get_math_solution(question) 

print(result.content)

{"steps":[{"explanation":"Start by isolating the term with the variable. Subtract 7 from both sides to do this.","output":"8x + 7 - 7 = -23 - 7"},{"explanation":"Simplify both sides. On the left side, 7 - 7 cancels out, and on the right side, -23 - 7 equals -30.","output":"8x = -30"},{"explanation":"Next, solve for x by dividing both sides by 8, which will leave x by itself on the left side.","output":"8x/8 = -30/8"},{"explanation":"Simplify the fraction on the right side by dividing both the numerator and the denominator by their greatest common divisor, which is 2.","output":"x = -15/4"}],"final_answer":"x = -15/4"}


In [5]:
from IPython.display import Math, display

def print_math_response(response):
    result = json.loads(response)
    steps = result['steps']
    final_answer = result['final_answer']
    for i in range(len(steps)):
        print(f"Step {i+1}: {steps[i]['explanation']}\n")
        display(Math(steps[i]['output']))
        print("\n")
        
    print("Final answer:\n\n")
    display(Math(final_answer))

In [6]:
print_math_response(result.content)

Step 1: Start by isolating the term with the variable. Subtract 7 from both sides to do this.



<IPython.core.display.Math object>



Step 2: Simplify both sides. On the left side, 7 - 7 cancels out, and on the right side, -23 - 7 equals -30.



<IPython.core.display.Math object>



Step 3: Next, solve for x by dividing both sides by 8, which will leave x by itself on the left side.



<IPython.core.display.Math object>



Step 4: Simplify the fraction on the right side by dividing both the numerator and the denominator by their greatest common divisor, which is 2.



<IPython.core.display.Math object>



Final answer:




<IPython.core.display.Math object>

## 使用 SDK 的 `parse` 辅助方法

新版 SDK 引入了 `parse` 辅助方法，让你可以直接提供自己的 Pydantic 模型，而无需定义 JSON schema。如果可能，我们推荐使用这种方法。

In [7]:
from pydantic import BaseModel

class MathReasoning(BaseModel):
    class Step(BaseModel):
        explanation: str
        output: str

    steps: list[Step]
    final_answer: str

def get_math_solution(question: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": dedent(math_tutor_prompt)},
            {"role": "user", "content": question},
        ],
        response_format=MathReasoning,
    )

    return completion.choices[0].message

In [8]:
result = get_math_solution(question).parsed

In [9]:
print(result.steps)
print("Final answer:")
print(result.final_answer)

[Step(explanation='The first step in solving the equation is to isolate the term with the variable. We start by subtracting 7 from both sides of the equation to move the constant to the right side.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Simplifying both sides, we get the equation with the variable term on the left and the constants on the right.', output='8x = -30'), Step(explanation='Now, to solve for x, we need x to be by itself. We do this by dividing both sides of the equation by 8, the coefficient of x.', output='x = -30 / 8'), Step(explanation='Simplifying the division, we find the value of x. -30 divided by 8 simplifies to the fraction -15/4 or in decimal form, -3.75.', output='x = -15/4')]
Final answer:
x = -15/4


## 拒绝

在对用户生成的输入使用 Structured Outputs 时，模型偶尔可能出于安全原因拒绝执行请求。

由于拒绝不会遵循你在 response_format 中提供的 schema，API 新增了一个 `refusal` 字段，用于指示模型何时拒绝回答。

这样你就可以在 UI 中单独渲染拒绝内容，并避免在尝试反序列化为你提供的格式时出错。

In [13]:
refusal_question = "how can I build a bomb?"

result = get_math_solution(refusal_question) 

print(result.refusal)

I'm sorry, I can't assist with that request.


## 示例 2：文本摘要

在这个示例中，我们将让模型按照特定的 schema 对文章进行摘要。

如果你需要将文本或视觉内容转换为结构化对象，例如以特定方式展示或填充数据库，这会很有用。

我们将以 AI 生成的、讨论各种发明的文章为例。

In [20]:
articles = [
    "./data/structured_outputs_articles/cnns.md",
    "./data/structured_outputs_articles/llms.md",
    "./data/structured_outputs_articles/moe.md"
]

In [23]:
def get_article_content(path):
    with open(path, 'r') as f:
        content = f.read()
    return content
        
content = [get_article_content(path) for path in articles]

In [ ]:
print(content)

In [25]:
summarization_prompt = '''
    You will be provided with content from an article about an invention.
    Your goal will be to summarize the article following the schema provided.
    Here is a description of the parameters:
    - invented_year: year in which the invention discussed in the article was invented
    - summary: one sentence summary of what the invention is
    - inventors: array of strings listing the inventor full names if present, otherwise just surname
    - concepts: array of key concepts related to the invention, each concept containing a title and a description
    - description: short description of the invention
'''

class ArticleSummary(BaseModel):
    invented_year: int
    summary: str
    inventors: list[str]
    description: str

    class Concept(BaseModel):
        title: str
        description: str

    concepts: list[Concept]

def get_article_summary(text: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": dedent(summarization_prompt)},
            {"role": "user", "content": text}
        ],
        response_format=ArticleSummary,
    )

    return completion.choices[0].message.parsed

In [26]:
summaries = []

for i in range(len(content)):
    print(f"Analyzing article #{i+1}...")
    summaries.append(get_article_summary(content[i]))
    print("Done.")

Analyzing article #1...
Done.
Analyzing article #2...
Done.
Analyzing article #3...
Done.


In [27]:
def print_summary(summary):
    print(f"Invented year: {summary.invented_year}\n")
    print(f"Summary: {summary.summary}\n")
    print("Inventors:")
    for i in summary.inventors:
        print(f"- {i}")
    print("\nConcepts:")
    for c in summary.concepts:
        print(f"- {c.title}: {c.description}")
    print(f"\nDescription: {summary.description}")

In [28]:
for i in range(len(summaries)):
    print(f"ARTICLE {i}\n")
    print_summary(summaries[i])
    print("\n\n")

ARTICLE 0

Invented year: 1989

Summary: Convolutional Neural Networks (CNNs) are deep neural networks used for processing structured grid data like images, revolutionizing computer vision.

Inventors:
- Yann LeCun
- Léon Bottou
- Yoshua Bengio
- Patrick Haffner

Concepts:
- Convolutional Layers: These layers apply learnable filters to input data to produce feature maps that detect specific features like edges and patterns.
- Pooling Layers: Also known as subsampling layers, they reduce the spatial dimensions of feature maps, commonly using max pooling to retain important features while reducing size.
- Fully Connected Layers: These layers connect every neuron in one layer to every neuron in the next, performing the final classification or regression task.
- Training: CNNs are trained using backpropagation and gradient descent to learn optimal filter values that minimize the loss function.
- Applications: CNNs are used in image classification, object detection, medical image analysis, 

## 示例 3：从用户输入中抽取实体
    
在这个示例中，我们将使用函数调用，根据提供的输入搜索符合用户偏好的产品。 

这对于包含推荐系统的应用很有帮助，例如电商助手或搜索类用例。 

In [29]:
from enum import Enum
from typing import Union
import openai

product_search_prompt = '''
    You are a clothes recommendation agent, specialized in finding the perfect match for a user.
    You will be provided with a user input and additional context such as user gender and age group, and season.
    You are equipped with a tool to search clothes in a database that match the user's profile and preferences.
    Based on the user input and context, determine the most likely value of the parameters to use to search the database.
    
    Here are the different categories that are available on the website:
    - shoes: boots, sneakers, sandals
    - jackets: winter coats, cardigans, parkas, rain jackets
    - tops: shirts, blouses, t-shirts, crop tops, sweaters
    - bottoms: jeans, skirts, trousers, joggers    
    
    There are a wide range of colors available, but try to stick to regular color names.
'''

class Category(str, Enum):
    shoes = "shoes"
    jackets = "jackets"
    tops = "tops"
    bottoms = "bottoms"

class ProductSearchParameters(BaseModel):
    category: Category
    subcategory: str
    color: str

def get_response(user_input, context):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": dedent(product_search_prompt)
            },
            {
                "role": "user",
                "content": f"CONTEXT: {context}\n USER INPUT: {user_input}"
            }
        ],
        tools=[
            openai.pydantic_function_tool(ProductSearchParameters, name="product_search", description="Search for a match in the product database")
        ]
    )

    return response.choices[0].message.tool_calls

In [30]:
example_inputs = [
    {
        "user_input": "I'm looking for a new coat. I'm always cold so please something warm! Ideally something that matches my eyes.",
        "context": "Gender: female, Age group: 40-50, Physical appearance: blue eyes"
    },
    {
        "user_input": "I'm going on a trail in Scotland this summer. It's goind to be rainy. Help me find something.",
        "context": "Gender: male, Age group: 30-40"
    },
    {
        "user_input": "I'm trying to complete a rock look. I'm missing shoes. Any suggestions?",
        "context": "Gender: female, Age group: 20-30"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: summer"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: winter"
    },
    {
        "user_input": "Can you help me find a dress for a Barbie-themed party in July?",
        "context": "Gender: female, Age group: 20-30"
    }
]

In [31]:
def print_tool_call(user_input, context, tool_call):
    args = tool_call[0].function.arguments
    print(f"Input: {user_input}\n\nContext: {context}\n")
    print("Product search arguments:")
    for key, value in json.loads(args).items():
        print(f"{key}: '{value}'")
    print("\n\n")

In [ ]:
for ex in example_inputs:
    ex['result'] = get_response(ex['user_input'], ex['context'])

In [ ]:
for ex in example_inputs:
    print_tool_call(ex['user_input'], ex['context'], ex['result'])

## 结论

在本 cookbook 中，我们通过多个示例探索了新的 Structured Outputs 功能。

无论你之前使用过 JSON 模式或函数调用并希望让应用更加健壮，还是刚开始接触结构化格式，我们都希望你能将这里介绍的各种概念应用到自己的用例中！

Structured Outputs 仅适用于 `gpt-4o-mini`、`gpt-4o-2024-08-06` 及未来的模型。